# Notebook 07 — Joint SPX + VIX Calibration

Calibrates Rough Heston parameters to match *both* the SPX IV surface
and the VIX futures term structure simultaneously using a combined loss.
Analyses weight sensitivity and parameter stability.

**Runtime estimate:** 10–20 min (grid search + yfinance)

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src") if os.path.basename(os.getcwd()) == "notebooks"
                else os.path.join(os.getcwd(), "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 11,
    "font.family": "serif",
})
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

import json
from datetime import date
from market.spx_data import download_spx_chain, clean_chain, to_iv_surface
from market.vix_futures import fetch_vix_futures
from market.vix_pricing import vix_futures_curve
from calibration.joint_calibration import calibrate_joint, calibrate_spx_only


## 1. Load Market Data

In [ ]:
VAL_DATE = date(2024, 1, 2)

df_spx  = download_spx_chain(VAL_DATE, cache=True)
df_clean = clean_chain(df_spx)
S0, R, Q = 4742.0, 0.053, 0.016   # SPX spot, risk-free rate, div yield
iv_obs   = to_iv_surface(df_clean, S0, R, Q)
print(f"SPX IV surface shape: {iv_obs.shape}")
print(f"ATM vol (6M): {iv_obs[2, 5]*100:.2f}%")

vix_df = fetch_vix_futures(VAL_DATE)
print(f"\nVIX futures ({len(vix_df)} contracts):")
print(vix_df[["expiry","settle_vix"]].to_string(index=False))


## 2. SPX-Only Baseline Calibration

In [ ]:
spx_result = calibrate_spx_only(iv_obs)
print("SPX-only calibration:")
rmse_key = next((k for k in ['spx_rmse_bps','rmse_bps','rmse'] if k in spx_result), None)
if rmse_key: print(f"  RMSE: {spx_result[rmse_key]:.1f} bps")
params_raw = spx_result.get('params', spx_result)
pnames = ['kappa','theta','sigma','rho','v0','H']
for k in pnames:
    if k in params_raw: print(f"  {k:6s} = {params_raw[k]:.4f}")


## 3. Joint SPX + VIX Calibration

In [ ]:
vix_spot = float(vix_df["settle_vix"].iloc[0]) if len(vix_df) else 13.5
joint_result = calibrate_joint(iv_obs, vix_spot, weights=(0.7, 0.3))
print("Joint SPX+VIX calibration:")
for key in ['spx_rmse_bps','rmse_bps','vix_error','total_loss','converged']:
    if key in joint_result: print(f"  {key}: {joint_result[key]}")
params_j = joint_result.get('params', joint_result)
for k in ['kappa','theta','sigma','rho','v0','H']:
    if k in params_j: print(f"  {k:6s} = {params_j[k]:.4f}")


## 4. Weight Sensitivity Analysis

Show how the calibrated parameters shift as we put more/less weight on VIX.

In [ ]:
weight_schemes = [
    (1.0, 0.0, "SPX only"),
    (1.0, 0.1, "SPX-heavy"),
    (1.0, 0.5, "Balanced"),
    (0.5, 1.0, "VIX-heavy"),
]
records = []
for w_spx, w_vix, label in weight_schemes:
    r = calibrate_joint(iv_obs, vix_spot, weights=(w_spx, w_vix))
    spx_r = r.get('spx_rmse_bps', r.get('rmse_bps', float('nan')))
    vix_r = r.get('vix_error', r.get('vix_rmse', float('nan')))
    rec = {"label": label, "w_vix": w_vix,
           "spx_rmse": spx_r, "vix_rmse": vix_r}
    rec.update(r.get("params", {}))
    records.append(rec)
    print(f"{label:12s}  SPX={spx_r:.1f}bps  VIX={vix_r:.4f}")

df_ws = pd.DataFrame(records)


## 5. Pareto Frontier: SPX RMSE vs VIX RMSE

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(df_ws["spx_rmse"], df_ws["vix_rmse"],
                c=df_ws["w_vix"], cmap="coolwarm", s=100, zorder=3)
for _, row in df_ws.iterrows():
    ax.annotate(row["label"], (row["spx_rmse"], row["vix_rmse"]),
                fontsize=8, textcoords="offset points", xytext=(6, 4))
plt.colorbar(sc, label="VIX weight w_vix")
ax.set_xlabel("SPX calibration RMSE (bps)")
ax.set_ylabel("VIX RMSE")
ax.set_title(f"Joint Calibration Pareto Frontier — {VAL_DATE}")
plt.tight_layout(); plt.show()


## 6. Model VIX Curve vs Market (Best Joint Fit)

In [ ]:
PKEYS = ['kappa','theta','sigma','rho','v0','H']
best_params = {k: joint_result[k] for k in PKEYS}
mats = np.linspace(1/12, 1.5, 50)
model_curve = vix_futures_curve(**best_params, maturities=mats)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(mats, model_curve, lw=2, label="Rough Heston (joint fit)")
if len(vix_df):
    t_mkt = np.arange(1, len(vix_df)+1) / 12
    ax.plot(t_mkt, vix_df["settle_vix"].values, "o--", ms=6, label="CBOE VIX futures")
ax.set_xlabel("Maturity (years)"); ax.set_ylabel("VIX")
ax.set_title(f"Model vs Market VIX Futures — {VAL_DATE}")
ax.legend(); plt.tight_layout(); plt.show()


## 7. SPX Residual Heatmap (Joint vs SPX-Only)

In [ ]:
# Reuse the model weights — normalizers already loaded by calibrate_joint above
from market.spx_data import T_GRID, K_GRID
from calibrate import _fno_predict_real_iv, _make_spatial_input
import calibrate as _cal
from fno_model import MirrorPaddedFNO2d
model7 = MirrorPaddedFNO2d(param_dim=6).to(DEVICE)
model7.load_state_dict(torch.load("../artifacts/weights/fno_v3_final_prod.pth", map_location=DEVICE))
model7.eval()

PKEYS = ['kappa','theta','sigma','rho','v0','H']

def get_pred(params_dict):
    pn, yn = _cal._param_norm, _cal._iv_norm
    p = np.array([params_dict[k] for k in PKEYS], dtype=np.float32)
    # _IdentityParamNorm has no transform; ParameterNormalizer does
    if hasattr(pn, 'transform'):
        p_norm = pn.transform(p.reshape(1, -1)).flatten().astype(np.float32)
    else:
        p_norm = p
    p_t = torch.tensor(p_norm).unsqueeze(0).to(DEVICE)
    spatial = _make_spatial_input(T_GRID, K_GRID, device=DEVICE)
    with torch.no_grad():
        iv_n = _fno_predict_real_iv(model7, p_t, spatial)
    raw = iv_n.squeeze().cpu().numpy()
    return yn.inverse_transform(raw) if hasattr(yn, 'inverse_transform') else raw

pred_joint = get_pred({k: joint_result[k] for k in PKEYS})
pred_spx   = get_pred({k: spx_result[k]   for k in PKEYS})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, pred, title in zip(axes,
        [pred_spx, pred_joint],
        ["SPX-only residuals (bps)", "Joint residuals (bps)"]):
    resid = (pred - iv_obs) * 1e4
    im = ax.imshow(resid, aspect="auto", cmap="RdBu_r", vmin=-30, vmax=30,
                   extent=[K_GRID[0], K_GRID[-1], T_GRID[-1], T_GRID[0]])
    ax.set_title(title); ax.set_xlabel("Log-moneyness"); ax.set_ylabel("Maturity (yr)")
    plt.colorbar(im, ax=ax, fraction=0.04, label="bps")
plt.suptitle("Calibration Residuals: SPX-only vs Joint")
plt.tight_layout(); plt.show()
